# Soccer Player Role + Team Inference

In [ ]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from detect_player import PlayerRoleTeamClassifier

print('Project root:', PROJECT_ROOT)

The first classifier construction may download PRTReID weights into `models/reid/` if they are missing.

In [ ]:
clf = PlayerRoleTeamClassifier.from_project_defaults(
    project_root=PROJECT_ROOT,
    device='cuda:0',  # uncomment when CUDA is available in this kernel
)

In [ ]:
sample_img = next((PROJECT_ROOT / 'data' / 'yolo' / 'fullframe' / 'images' / 'val').glob('*.jpg'))
output_dir = PROJECT_ROOT / 'outputs' / 'detect_player' / 'infer' / 'team_classifier'
output_dir.mkdir(parents=True, exist_ok=True)

results = clf.predict(sample_img)
json_path = output_dir / f'{sample_img.stem}.json'
image_path = output_dir / f'{sample_img.stem}.jpg'

clf.save_json(results, json_path)
annotated = clf.draw(sample_img, results, output_path=image_path)

print('Image:', sample_img)
print('Detections:', len(results))
print('JSON:', json_path)
print('Annotated:', image_path)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(image_path)))


## Folder image inference

Use this when your input is a directory of frames/images instead of a video.

In [ ]:
folder = PROJECT_ROOT / "data" / "raw" / "tracking" / "train" / "SNMOT-060" / "img1"
output_video = output_dir / "SNMOT-060_team_classifier.mp4"
output_json = output_dir / "SNMOT-060_team_classifier.json"

folder_summary = clf.predict_folder(
    folder,
    output_video_path=output_video,
    output_json_path=output_json,
    output_fps=25.0,
    progress=True,
)
folder_summary
